# GOAT-Evolution — TRL SFT fine-tune (Colab)

Hugging Face TRL `SFTTrainer` + PEFT/LoRA on a 2B/3B Hugging Face model, trained on per-attack JSONL shards from the GOAT-Evolution environment.

Two ways to feed data:

1. **Generate fresh shards** by playing episodes in the env with a Gemini judge (recommended for the demo).
2. **Download an existing dataset repo** that you already pushed.

Outputs: trained LoRA adapter, reward + loss curves, baseline-vs-trained eval, optional push to Hugging Face.

## 1. Install

In [ ]:
%pip install -q -U \
    "transformers>=4.44" \
    "trl>=0.13" \
    "peft>=0.12" \
    "datasets>=2.20" \
    "accelerate>=0.34" \
    "huggingface_hub>=0.26" \
    "openenv-core>=0.2.1" \
    "google-generativeai>=0.8.3" \
    "matplotlib>=3.8" \
    "pydantic>=2.9" \
    "fastapi>=0.115" \
    "sse-starlette>=2.1" \
    "httpx>=0.27"

## 2. Clone the repo

In [ ]:
import os, sys, subprocess, pathlib
REPO_URL = os.environ.get('GOAT_REPO_URL', 'https://github.com/YOUR_USER/goat-evolution.git')
REPO_DIR = pathlib.Path('/content/goat-evolution')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=False)
PROJECT = REPO_DIR / 'goat_evolution' if (REPO_DIR / 'goat_evolution').exists() else REPO_DIR
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)
print('cwd:', os.getcwd())

## 3. Secrets

In [ ]:
import os
try:
    from google.colab import userdata
    for key in ('HF_TOKEN', 'GEMINI_API_KEY'):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except Exception:
    pass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = input('Paste HF_TOKEN: ').strip()
from huggingface_hub import login
login(os.environ['HF_TOKEN'])
if not os.environ.get('GEMINI_API_KEY'):
    print('GEMINI_API_KEY missing -> use scripted providers for local data generation.')

## 4. Configuration

In [ ]:
MODEL_BASE      = 'Qwen/Qwen2.5-3B-Instruct'  # try Qwen/Qwen2.5-1.5B-Instruct on small GPU
MODEL_OUT       = 'YOUR_USER/goat-evolution-3b-defender'
DATASET_REPO    = ''                           # leave blank to generate shards locally
EPISODES_TO_GEN = 3                            # quick default for smoke runs
ARTIFACTS_DIR   = '/content/artifacts'
OUT_DIR         = '/content/sft_out'
EPOCHS          = 1
BATCH_SIZE      = 1
GRAD_ACCUM      = 8
LR              = 2e-4
MAX_LEN         = 768
LORA_R          = 8
PUSH            = False                         # True to push the adapter at the end
ONLY_BREAKING   = True                          # Only train on breaking attacks (best signal)
RED_PROVIDER    = 'scripted'                    # switch to gemini if API key exists
BLUE_PROVIDER   = 'scripted'
JUDGE_PROVIDER  = 'scripted'
print({'MODEL_BASE': MODEL_BASE, 'episodes_to_gen': EPISODES_TO_GEN, 'providers': (RED_PROVIDER, BLUE_PROVIDER, JUDGE_PROVIDER)})

## 5. Get the per-attack dataset

Either generate it on the fly from the env, or download an existing dataset repo.

In [ ]:
import asyncio, pathlib, os
from backend.cli.realtime_trainer import RealtimeTrainer
from backend.training.llm_factory import build_agents
from backend.training.logging_setup import setup_logging

if DATASET_REPO:
    from huggingface_hub import snapshot_download
    data_dir = snapshot_download(repo_id=DATASET_REPO, repo_type='dataset')
    DATASET_JSONL = pathlib.Path(data_dir) / 'realtime_samples.jsonl'
    if not DATASET_JSONL.exists():
        # combine shards if master file isn't present
        shards = sorted(pathlib.Path(data_dir).glob('shards/*.jsonl'))
        DATASET_JSONL = pathlib.Path('/content/combined.jsonl')
        DATASET_JSONL.write_text(''.join(p.read_text() for p in shards))
else:
    setup_logging(logs_root='/content/logs', run_label='trl')
    if (RED_PROVIDER == 'gemini' or BLUE_PROVIDER == 'gemini' or JUDGE_PROVIDER == 'gemini') and not os.environ.get('GEMINI_API_KEY'):
        print('GEMINI_API_KEY missing; forcing scripted providers.')
        RED_PROVIDER = BLUE_PROVIDER = JUDGE_PROVIDER = 'scripted'
    agents = build_agents(
        red_provider_name=RED_PROVIDER,
        blue_provider_name=BLUE_PROVIDER,
        judge_provider_name=JUDGE_PROVIDER,
    )
    trainer = RealtimeTrainer(
        artifacts_root=pathlib.Path(ARTIFACTS_DIR),
        subject_size='3b',
        backend='stub',  # use stub during data-gen; we train the real model below
        push_each_attack=False,
        hf_dataset_repo=None,
        hf_model_repo=None,
        gate_min_violation_drop=0.0,
        gate_max_refusal_drop=0.05,
        learning_rate=5e-5,
        agents=agents,
    )
    async def _gen():
        for i in range(EPISODES_TO_GEN):
            r = await trainer.run_episode(task_id=4, model_profile='standard_model')
            print(f'  ep {i+1}: turns={r.turns} breaking={r.breaking_attacks}')
    asyncio.run(_gen())
    DATASET_JSONL = pathlib.Path(ARTIFACTS_DIR) / 'dataset' / 'realtime_samples.jsonl'
print('Dataset:', DATASET_JSONL, '->', DATASET_JSONL.stat().st_size, 'bytes')

## 6. Load and tokenize the per-attack dataset

In [ ]:
import json
from datasets import Dataset
rows = []
with open(DATASET_JSONL) as fh:
    for line in fh:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        if ONLY_BREAKING and not rec.get('is_breaking'):
            continue
        rows.append(rec)
print(f'Training rows: {len(rows)}')
def to_text(row):
    prompt = row.get('prompt', '')
    chosen = row.get('chosen_response', '')
    return {'text': f'<|user|>\n{prompt}\n<|assistant|>\n{chosen}'}
dataset = Dataset.from_list([to_text(r) for r in rows])
dataset

## 7. TRL SFTTrainer with LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model
from trl import SFTConfig, SFTTrainer
tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_BASE, torch_dtype=torch.bfloat16)
lora = LoraConfig(r=LORA_R, lora_alpha=LORA_R*2, lora_dropout=0.05, target_modules=['q_proj','v_proj'], task_type=TaskType.CAUSAL_LM)
model = get_peft_model(model, lora)
model.print_trainable_parameters()
args = SFTConfig(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    max_length=MAX_LEN,
    logging_steps=2,
    save_steps=50,
    save_total_limit=2,
    bf16=torch.cuda.is_available(),
    report_to=[],
    push_to_hub=PUSH,
    hub_model_id=MODEL_OUT if PUSH else None,
)
trainer_sft = SFTTrainer(model=model, args=args, train_dataset=dataset, processing_class=tokenizer)
result = trainer_sft.train()
trainer_sft.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print('Train metrics:', result.metrics)

## 8. Plot loss curve

In [ ]:
import matplotlib.pyplot as plt
history = trainer_sft.state.log_history
steps  = [h['step'] for h in history if 'loss' in h]
losses = [h['loss'] for h in history if 'loss' in h]
plt.figure(figsize=(8,4))
plt.plot(steps, losses, label='train loss')
plt.xlabel('training step'); plt.ylabel('loss'); plt.title('TRL SFT loss curve')
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/loss_curve.png', dpi=130); plt.show()

## 9. Baseline vs trained on the GOAT-Evolution holdout

In [ ]:
import pathlib, torch
from backend.training.gating import HoldoutGate
from backend.training.llm_subject import LLMSubject
trainer_sft.model.eval()
subject_trained = LLMSubject(backend='hf_local', model_name=MODEL_BASE, adapter_dir=pathlib.Path(OUT_DIR))
subject_trained._tokenizer = tokenizer
subject_trained._model = trainer_sft.model
subject_trained._hf_loaded = True
subject_baseline = LLMSubject(backend='stub', model_name=MODEL_BASE)
gate = HoldoutGate()
before = gate.evaluate(subject_baseline)
after  = gate.evaluate(subject_trained)
print('Before:', before.to_dict()); print('After :', after.to_dict())

In [ ]:
import matplotlib.pyplot as plt
labels = ['asr (lower=better)', 'avg_violation', 'refusal_quality']
b = [before.asr, before.avg_violation, before.refusal_quality]
a = [after.asr,  after.avg_violation,  after.refusal_quality]
x = range(len(labels))
plt.figure(figsize=(7,4))
plt.bar([i-0.18 for i in x], b, width=0.36, label='baseline')
plt.bar([i+0.18 for i in x], a, width=0.36, label='trained')
plt.xticks(list(x), labels); plt.ylabel('score'); plt.title('Holdout: baseline vs trained')
plt.legend(); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/holdout_compare.png', dpi=130); plt.show()

## 10. Push the adapter (optional)

In [ ]:
if PUSH:
    trainer_sft.push_to_hub(MODEL_OUT)
    tokenizer.push_to_hub(MODEL_OUT)
    print('Pushed adapter:', f'https://huggingface.co/{MODEL_OUT}')